# Lab 1 — Transfer Learning untuk Detection (Faster R-CNN)
Fine-tuning `fasterrcnn_resnet50_fpn` (pre-trained COCO) supaya hanya mendeteksi **Person** dan **Bicycle** pada Pascal VOC 2012.

Jalankan sel demi sel di Google Colab. Aktifkan GPU: `Runtime > Change runtime type > GPU`.

In [ ]:
!pip -q install torchmetrics
import torch, torchvision
print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

In [ ]:
import torch, numpy as np
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.datasets import VOCDetection
import torchvision.transforms.functional as F
from torch.utils.data import DataLoader, Subset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hanya 2 kelas target + background
TARGET_CLASSES = ['background', 'person', 'bicycle']
CLASS2IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
NUM_CLASSES = len(TARGET_CLASSES)  # 3 (termasuk background)

## Dataset: wrapper VOC yang hanya mengambil objek person & bicycle

In [ ]:
class VOCPersonBike(torch.utils.data.Dataset):
    def __init__(self, root, image_set='train', download=True):
        self.base = VOCDetection(root=root, year='2012', image_set=image_set, download=download)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]
        objs = target['annotation'].get('object', [])
        if isinstance(objs, dict):
            objs = [objs]

        boxes, labels = [], []
        for obj in objs:
            name = obj['name']
            if name not in ('person', 'bicycle'):
                continue
            bb = obj['bndbox']
            xmin, ymin = float(bb['xmin']), float(bb['ymin'])
            xmax, ymax = float(bb['xmax']), float(bb['ymax'])
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS2IDX[name])

        img = F.to_tensor(img)
        if len(boxes) == 0:
            # Gambar tanpa objek target -> tetap dikembalikan dengan box kosong
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
            labels_t = torch.as_tensor(labels, dtype=torch.int64)

        targets = {
            'boxes': boxes_t,
            'labels': labels_t,
            'image_id': torch.tensor([idx]),
        }
        return img, targets


def collate_fn(batch):
    return tuple(zip(*batch))


def filter_has_object(ds):
    """Buang gambar yang tidak punya person/bicycle sama sekali (mempercepat training)."""
    keep = []
    for i in range(len(ds)):
        _, t = ds[i]
        if t['boxes'].shape[0] > 0:
            keep.append(i)
    return Subset(ds, keep)

In [ ]:
train_full = VOCPersonBike(root='./data', image_set='train', download=True)
val_full   = VOCPersonBike(root='./data', image_set='val', download=True)

print('Memfilter gambar tanpa objek target (bisa memakan waktu beberapa menit)...')
train_ds = filter_has_object(train_full)
val_ds   = filter_has_object(val_full)
print(f'Train: {len(train_ds)} gambar | Val: {len(val_ds)} gambar')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

## Model: ganti box predictor head agar sesuai 2 kelas + background

In [ ]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

## Loop training + evaluasi mAP dengan torchmetrics

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

EPOCHS = 5

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for imgs, targets in loader:
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_map(model, loader, device):
    model.eval()
    metric = MeanAveragePrecision()
    for imgs, targets in loader:
        imgs = [img.to(device) for img in imgs]
        preds = model(imgs)
        preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: v for k, v in t.items() if k in ('boxes', 'labels')} for t in targets]
        metric.update(preds, targets_cpu)
    result = metric.compute()
    return result


history = {'train_loss': [], 'val_map': []}
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    lr_scheduler.step()
    map_result = evaluate_map(model, val_loader, DEVICE)
    history['train_loss'].append(loss)
    history['val_map'].append(map_result['map'].item())
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {loss:.4f} | mAP@[.5:.95]: {map_result['map']:.4f} | mAP@50: {map_result['map_50']:.4f}")

## Plot hasil training

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss'], marker='o'); axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_map'], marker='s', color='green'); axes[1].set_title('Val mAP'); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.savefig('lab1_history.png', dpi=150); plt.show()

## Visualisasi prediksi pada satu gambar validasi

In [ ]:
import matplotlib.patches as patches

model.eval()
img, target = val_ds[0]
with torch.no_grad():
    pred = model([img.to(DEVICE)])[0]

fig, ax = plt.subplots(1, figsize=(6, 6))
ax.imshow(img.permute(1, 2, 0).numpy())

for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
    if score < 0.5:
        continue
    x1, y1, x2, y2 = box.cpu().numpy()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f"{TARGET_CLASSES[label]} {score:.2f}", color='lime', fontsize=10)

plt.axis('off'); plt.show()

In [ ]:
torch.save(model.state_dict(), 'fasterrcnn_person_bicycle.pth')
print('Model tersimpan: fasterrcnn_person_bicycle.pth')